In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from data.combine_timetable import combine_timetables
from data.timetable import load_gold, save_gold


In [2]:
n_freight_trains = 182

passenger = load_gold('passenger')
freight   = load_gold('freight', n_trains=n_freight_trains)

# Validatie
overlap = set(passenger['TRAIN_NO']) & set(freight['TRAIN_NO'])
print(f"Overlappende TRAIN_NO: {overlap if overlap else 'geen'}")

combined = combine_timetables(n_freight_trains)
print(combined['TRAIN_TYPE'].value_counts())
print(f"ENTRY_SECONDS range: {combined['ENTRY_SECONDS'].min():.0f}s — {combined['ENTRY_SECONDS'].max():.0f}s")

Overlappende TRAIN_NO: geen
TRAIN_TYPE
IC         6158
L          3275
freight     654
EURST       324
INT         189
ICE         126
Name: count, dtype: int64
ENTRY_SECONDS range: 0s — 86941s


In [3]:
combined.head()

,SECTION,SOURCE,TARGET,PLANNED_ENTRY,PLANNED_EXIT,TRAIN_NO,TYPE,TRAIN_TYPE,PERIOD,DYNAMICS,ENTRY_SECONDS,EXIT_SECONDS
0,36N:SCHAARBEEK-BRUSSEL-NOORD,SCHAARBEEK,BRUSSEL-NOORD,2025-01-01 21:23:00,2025-01-01 21:26:00,10,SOURCE,ICE,EVENING,ACC-BR,76860,77040
1,BRUSSEL-NOORD -- platform 3,BRUSSEL-NOORD,BRUSSEL-NOORD,2025-01-01 21:26:00,2025-01-01 21:28:00,10,WITHIN-STATION-DWELL,ICE,EVENING,0-0,77040,77160
2,0-2:BRUSSEL-NOORD-BRUSSEL-CONGRES,BRUSSEL-NOORD,BRUSSEL-CONGRES,2025-01-01 21:28:00,2025-01-01 21:30:00,10,BETWEEN-STATION,ICE,EVENING,ACC-0,77160,77280
3,BRUSSEL-CONGRES -- platform 2,BRUSSEL-CONGRES,BRUSSEL-CONGRES,2025-01-01 21:30:00,2025-01-01 21:30:01,10,WITHIN-STATION-PASSING,ICE,EVENING,0-0,77280,77281
4,0-2:BRUSSEL-CONGRES-BRUSSEL-CENTRAAL,BRUSSEL-CONGRES,BRUSSEL-CENTRAAL,2025-01-01 21:30:01,2025-01-01 21:31:00,10,BETWEEN-STATION,ICE,EVENING,0-0,77281,77340


In [4]:
save_gold(combined, source='combined', n_trains=n_freight_trains)

Opgeslagen (combined): 1210 treinen, 10726 segmenten → /Users/ddw/Desktop/Rescheduling/data/gold/combined/182


In [5]:
# =============================================================================
# Vergelijk running times tussen train types op meerdere segmenten
# =============================================================================

import matplotlib.pyplot as plt

# Running time toevoegen
df['RUNNING_TIME'] = (
    df['EXIT_SECONDS'] - df['ENTRY_SECONDS']
)

# Enkel between-station
between = df[df['TYPE'] == 'BETWEEN-STATION'].copy()

# Zoek segmenten met meerdere train types
candidate_sections = []

for section, subset in between.groupby('SECTION'):

    counts = subset.groupby('TRAIN_TYPE').size()

    # minstens 2 types
    if len(counts) < 2:
        continue

    # minstens 30 observaties per type
    if (counts >= 30).all():

        score = counts.sum()

        candidate_sections.append({
            'section': section,
            'n_types': len(counts),
            'n_obs': score
        })

# Sorteer op meeste data + meeste types
candidate_sections = sorted(
    candidate_sections,
    key=lambda x: (x['n_types'], x['n_obs']),
    reverse=True
)

print(f"Gevonden segmenten: {len(candidate_sections)}")

# ============================================================================
# Visualiseer top N segmenten
# ============================================================================

TOP_N = 5

for info in candidate_sections[:TOP_N]:

    section = info['section']

    print("=" * 80)
    print(f"SECTION: {section}")
    print("=" * 80)

    segment_df = between[
        between['SECTION'] == section
    ].copy()

    # ------------------------------------------------------------------------
    # Stats
    # ------------------------------------------------------------------------

    stats = (
        segment_df.groupby('TRAIN_TYPE')['RUNNING_TIME']
        .agg(['count', 'mean', 'std', 'median'])
        .sort_values('mean')
    )

    display(stats)

    # ------------------------------------------------------------------------
    # Histogram
    # ------------------------------------------------------------------------

    plt.figure(figsize=(12, 5))

    for ttype in sorted(segment_df['TRAIN_TYPE'].unique()):

        subset = segment_df[
            segment_df['TRAIN_TYPE'] == ttype
        ]['RUNNING_TIME']

        plt.hist(
            subset,
            bins=20,
            alpha=0.5,
            density=True,
            label=f"{ttype} (n={len(subset)})"
        )

    plt.xlabel("Running time (s)")
    plt.ylabel("Density")
    plt.title(f"Running times per train type\n{section}")
    plt.legend()
    plt.show()

    # ------------------------------------------------------------------------
    # Boxplot
    # ------------------------------------------------------------------------

    plot_data = []
    labels = []

    for ttype in sorted(segment_df['TRAIN_TYPE'].unique()):

        subset = segment_df[
            segment_df['TRAIN_TYPE'] == ttype
        ]['RUNNING_TIME']

        plot_data.append(subset)
        labels.append(ttype)

    plt.figure(figsize=(10, 5))

    plt.boxplot(
        plot_data,
        tick_labels=labels,
        showfliers=False
    )

    plt.ylabel("Running time (s)")
    plt.title(f"Running time spread per train type\n{section}")

    plt.show()

NameError: name 'df' is not defined

In [ ]:
print(df.columns)

Index(['SECTION', 'SOURCE', 'TARGET', 'PLANNED_ENTRY', 'PLANNED_EXIT',
       'TRAIN_NO', 'TYPE', 'TRAIN_TYPE', 'PERIOD', 'DYNAMICS', 'ENTRY_SECONDS',
       'EXIT_SECONDS', 'RUNNING_TIME'],
      dtype='str')


In [8]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from data.timetable import load_gold, save_gold

df = load_gold("combined", 182)

In [9]:
for section in sorted(df['SECTION'].unique()):
    print(section)

0-1:BRUSSEL-CENTRAAL-BRUSSEL-CONGRES
0-1:BRUSSEL-CONGRES-BRUSSEL-NOORD
0-1:BRUSSEL-KAPELLEKERK-BRUSSEL-CENTRAAL
0-1:BRUSSEL-ZUID-BRUSSEL-KAPELLEKERK
0-2:BRUSSEL-CENTRAAL-BRUSSEL-KAPELLEKERK
0-2:BRUSSEL-CONGRES-BRUSSEL-CENTRAAL
0-2:BRUSSEL-KAPELLEKERK-BRUSSEL-ZUID
0-2:BRUSSEL-NOORD-BRUSSEL-CONGRES
0-2:BRUSSEL-ZUID-BRUSSEL-KAPELLEKERK
0-3:BRUSSEL-CENTRAAL-BRUSSEL-CONGRES
0-3:BRUSSEL-CONGRES-BRUSSEL-NOORD
0-3:BRUSSEL-KAPELLEKERK-BRUSSEL-CENTRAAL
0-3:BRUSSEL-ZUID-BRUSSEL-KAPELLEKERK
0-4:BRUSSEL-CENTRAAL-BRUSSEL-KAPELLEKERK
0-4:BRUSSEL-CONGRES-BRUSSEL-CENTRAAL
0-4:BRUSSEL-KAPELLEKERK-BRUSSEL-ZUID
0-4:BRUSSEL-NOORD-BRUSSEL-CONGRES
0-5:BRUSSEL-CENTRAAL-BRUSSEL-CONGRES
0-5:BRUSSEL-CONGRES-BRUSSEL-NOORD
0-5:BRUSSEL-KAPELLEKERK-BRUSSEL-CENTRAAL
0-5:BRUSSEL-ZUID-BRUSSEL-KAPELLEKERK
0-6:BRUSSEL-CENTRAAL-BRUSSEL-KAPELLEKERK
0-6:BRUSSEL-CONGRES-BRUSSEL-CENTRAAL
0-6:BRUSSEL-KAPELLEKERK-BRUSSEL-ZUID
0-6:BRUSSEL-NOORD-BRUSSEL-CONGRES
124:BRUSSEL-ZUID-VORST-OOST
124:VORST-OOST-BRUSSEL-ZUID
161-2:BRUSSEL